# Análise de Desempenho: Paralelização de Equações Normais

**Disciplina:** INF01008 - Programação Paralela  

**Plataforma de Testes:** Cluster GPPD (Hype)  

## 0. VTune Parsing

In [ ]:
import os
import re
import pandas as pd
import numpy as np

def parse_vtune_value(value_str: str) -> float:
    """Converte strings do VTune para float, incluindo '60.1% (...)' e '7,475,000,000'."""
    if not value_str:
        return 0.0
    try:
        # Remove separador de milhar, unidade %, espaços e tudo após o primeiro token
        cleaned = value_str.strip().replace(',', '').split()[0].replace('%', '')
        return float(cleaned)
    except (ValueError, IndexError):
        return 0.0

def get_dram_bandwidth_from_hpc(file_path: str) -> dict:
    """
    Extrai os dados reais de bandwidth DRAM em GB/s do HPC.
    Linha: 1 [TAB] DRAM, GB/sec [TAB] platform_max [TAB] observed_max [TAB] average [TAB] ...
    Retorna platform_max, observed_max e average.
    DRAM Bound é N/A com HT ativo (Haswell + HT) — usar GB/s é a única opção.
    """
    result = {'DRAM GB/s Platform Max': 0.0,
              'DRAM GB/s Observed Max': 0.0,
              'DRAM GB/s Average':      0.0}
    if not os.path.exists(file_path):
        return result
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = [p.strip() for p in line.split('\t')]
                if len(parts) >= 5 and 'DRAM, GB/sec' in parts[1]:
                    result['DRAM GB/s Platform Max'] = parse_vtune_value(parts[2])
                    result['DRAM GB/s Observed Max'] = parse_vtune_value(parts[3])
                    result['DRAM GB/s Average']      = parse_vtune_value(parts[4])
                    break
    except Exception as e:
        print(f"⚠️ Erro ao ler bandwidth: {e}")
    return result

def get_hotspot_function(file_path):
    """Extrai o nome da função que mais consumiu tempo (Top Hotspot)."""
    if not os.path.exists(file_path):
        return "N/A", 0.0
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            if "Function" in line and "Module" in line:
                target_line = lines[i+1]
                parts = [p.strip() for p in target_line.split('\t')]
                if len(parts) >= 4:
                    func_name = parts[1]
                    cpu_time = parse_vtune_value(parts[3])
                    return func_name, cpu_time
        return "N/A", 0.0
    except:
        return "Error", 0.0

def get_metrics_from_csv(file_path: str, metrics_list: list) -> dict:
    """
    Parser genérico para arquivos VTune (snap, hot).
    Procura o nome da métrica em QUALQUER coluna da linha
    e pega o valor na coluna seguinte.
    """
    results = {m: 0.0 for m in metrics_list}
    if not os.path.exists(file_path):
        return results
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = [p.strip() for p in line.split('\t')]
                for col_idx, part in enumerate(parts):
                    for m in metrics_list:
                        if part.lower() == m.lower():
                            for next_col in parts[col_idx + 1:]:
                                if next_col:
                                    results[m] = parse_vtune_value(next_col)
                                    break
    except Exception as e:
        print(f"⚠️ Erro ao ler {file_path}: {e}")
    return results

def get_hpc_metrics_isolated(file_path, metrics_list):
    """
    Parser específico para arquivos HPC do VTune.
    Assume estrutura: Hierarchy Level [TAB] Metric Name [TAB] Metric Value
    """
    results = {m: 0.0 for m in metrics_list}
    if not os.path.exists(file_path):
        return results
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = [p.strip() for p in line.split('\t')]
                if len(parts) >= 3:
                    name_file = parts[1].lower()
                    value_file = parts[2]
                    for m in metrics_list:
                        if m.lower() == name_file:
                            results[m] = parse_vtune_value(value_file)
        return results
    except Exception as e:
        print(f"⚠️ Erro na leitura física do arquivo: {e}")
        return results

print("✅ Parsers carregados.")

## 1. Baselines

### 1.1. Data Load

In [ ]:
PATH_BASE     = "./baselines_20260418_1847/app_bench_results.csv"
DIR_BASE_VTUNE = "./baselines_20260418_1847/vtune_csvs"

def load_baseline(path):
    df = pd.read_csv(path)
    df = df[~((df["samples"] == 515345) & (df["features"] == 90))]
    df['opt_level'] = np.where(df.index % 6 < 3, 'O0 (Raw)', 'O3 (Opt)')
    df['threads'] = 1
    group_cols = ['dataset', 'samples', 'features', 'threads', 'opt_level']
    df_mean = df.groupby(group_cols).mean(numeric_only=True).reset_index()
    df_mean['problem_size'] = df_mean.apply(
        lambda x: f"{int(x['samples'])} x {int(x['features'])}", axis=1)
    return df_mean.sort_values(['samples', 'features'])

df_baseline = load_baseline(PATH_BASE)
problem_order = df_baseline.sort_values(['samples','features'])['problem_size'].unique().tolist()

print("✅ Baseline carregado.")
display(df_baseline[['problem_size', 'opt_level', 'fit_time']])

### 1.2. Baseline Time

In [ ]:
import altair as alt

baseline_viz = alt.Chart(df_baseline).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('fit_time:Q', title='Tempo (s)'),
    color=alt.Color('opt_level:N', legend=alt.Legend(title="Otimização")),
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema (n x m)'),
    tooltip=['problem_size', 'opt_level', 'fit_time']
).properties(width=80, title='Desempenho Sequencial: O0 vs O3')

baseline_viz

### 1.4. Hotspot

In [ ]:
hot_metrics = ['CPU Time', 'CPI Rate', 'Instructions Retired']

hotspot_list = []
for _, row in df_baseline.iterrows():
    ds_id   = row['dataset'].replace('.csv', '')
    opt_tag = 'raw' if 'O0' in row['opt_level'] else 'opt'
    fpath   = f"{DIR_BASE_VTUNE}/hot_{opt_tag}_{ds_id}.csv"
    m_results = get_metrics_from_csv(fpath, hot_metrics)
    top_func, top_func_time = get_hotspot_function(fpath)
    hotspot_list.append({
        'problem_size': row['problem_size'],
        'samples':      row['samples'],
        'opt_level':    row['opt_level'],
        'top_function': top_func,
        'func_cpu_time': top_func_time,
        **m_results
    })

df_hotspots = pd.DataFrame(hotspot_list)
print("✅ Hotspots carregados.")
display(df_hotspots[['problem_size','opt_level','top_function','CPI Rate','CPU Time','Instructions Retired']])

In [ ]:
# --- Fator de redução de instruções O3 vs O0 ---
# Este gráfico é a prova direta do efeito SIMD:
# O3/AVX2 retira ~10-12x menos instruções para o mesmo trabalho.
# Isso explica por que O3 satura a memória mais rápido:
# cada instrução move mais dados (8 doubles/instrução vs 1).

df_hot_plot = df_hotspots[df_hotspots['CPU Time'] > 0].copy()

df_hot_o0 = df_hot_plot[df_hot_plot['opt_level']=='O0 (Raw)'][['problem_size','samples','Instructions Retired']]\
    .rename(columns={'Instructions Retired': 'inst_o0'})
df_hot_o3 = df_hot_plot[df_hot_plot['opt_level']=='O3 (Opt)'][['problem_size','samples','Instructions Retired']]\
    .rename(columns={'Instructions Retired': 'inst_o3'})

df_inst_ratio = pd.merge(df_hot_o0, df_hot_o3, on=['problem_size','samples'])
df_inst_ratio['Reduction Factor'] = df_inst_ratio['inst_o0'] / df_inst_ratio['inst_o3']

print("Fator de redução de instruções (O0 / O3) — quanto menor O3 retira:")
display(df_inst_ratio[['problem_size','inst_o0','inst_o3','Reduction Factor']])

inst_chart = alt.Chart(df_inst_ratio).mark_bar(color='steelblue').encode(
    x=alt.X('problem_size:N', sort=problem_order, title='Tamanho do Problema', axis=alt.Axis(labelAngle=-25)),
    y=alt.Y('Reduction Factor:Q', title='Fator de Redução (O0 / O3)'),
    tooltip=['problem_size','Reduction Factor']
).properties(
    width=500, height=300,
    title='Redução de Instruções por SIMD (O3 vs O0) — prova da vetorização AVX'
)

inst_chart

In [ ]:
df_plot_hot = df_hot_plot.copy()

chart_cpu_time = alt.Chart(df_plot_hot).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('CPU Time:Q', scale=alt.Scale(type='log'), title='Tempo de CPU (s) - Escala Log'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','opt_level','CPU Time']
).properties(width=80, title='Redução de Tempo: O0 vs O3')

chart_cpi = alt.Chart(df_plot_hot).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('CPI Rate:Q', title='CPI Rate (Menor é melhor)'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','opt_level','CPI Rate']
).properties(width=80, title='Eficiência do Pipeline (CPI)')

(chart_cpu_time & chart_cpi).resolve_scale(y='independent')

### 1.5. HPC

In [ ]:

hpc_targets = ['CPI Rate', 'Memory Bound', 'Cache Bound',
               'Vectorization Intensity', 'Effective Physical Core Utilization']

hpc_baseline_rows = []
for _, row in df_baseline.iterrows():
    ds_id   = row['dataset'].replace('.csv', '')
    opt_tag = 'raw' if 'O0' in row['opt_level'] else 'opt'
    fpath   = f"{DIR_BASE_VTUNE}/hpc_{opt_tag}_{ds_id}.csv"
    
    # Coleta as métricas gerais
    m_data  = get_hpc_metrics_isolated(fpath, hpc_targets)
    # ATUALIZAÇÃO: Agora chamamos a função que a IA esqueceu de usar
    bw_data = get_dram_bandwidth_from_hpc(fpath)
    
    hpc_baseline_rows.append({
        'problem_size': row['problem_size'],
        'opt_level':    row['opt_level'],
        'samples':      row['samples'],
        **m_data,
        **bw_data
    })

df_hpc_baseline = pd.DataFrame(hpc_baseline_rows)
print("📊 Métricas HPC Baseline:")
display(df_hpc_baseline[['problem_size','opt_level','CPI Rate','Memory Bound',
                          'Cache Bound','DRAM GB/s Average','Vectorization Intensity']])

In [ ]:
# Gráfico de Memory Wall Baseline - Corrigido
df_hpc_large = df_hpc_baseline[df_hpc_baseline['samples'] >= 250000].copy()

mem_base = alt.Chart(df_hpc_large).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('Memory Bound:Q', title='Memory Bound (%)'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','opt_level','Memory Bound']
).properties(width=80, title='Memory Bound: O0 vs O3')

# ATUALIZAÇÃO: Gráfico de Bandwidth em GB/s
dram_base = alt.Chart(df_hpc_large).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('DRAM GB/s Average:Q', title='DRAM Average (GB/s)'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','opt_level','DRAM GB/s Average']
).properties(width=80, title='Tráfego de RAM (GB/s): Prova do Memory Wall')

vec_base = alt.Chart(df_hpc_large).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('Vectorization Intensity:Q', title='Vectorization Intensity'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','opt_level','Vectorization Intensity']
).properties(width=80, title='Vectorization Intensity: confirma AVX em O3')

(mem_base & dram_base & vec_base).resolve_scale(y='independent')

## 2. Threads

### 2.1. Load data

In [ ]:
PATH_PARA      = "./results_20260418_1749/app_bench_results.csv"
DIR_PARA_VTUNE = "./results_20260418_1749/vtune_csvs"

df_para_raw = pd.read_csv(PATH_PARA)
group_cols  = ['dataset', 'samples', 'features', 'threads']
df_parallel = df_para_raw.groupby(group_cols).mean(numeric_only=True).reset_index()
df_parallel['problem_size'] = df_parallel.apply(
    lambda x: f"{int(x['samples'])} x {int(x['features'])}", axis=1)
df_parallel = df_parallel.sort_values(['samples', 'features', 'threads'])

print(f"✅ Dados paralelos carregados! ({len(df_para_raw)} amostras originais)")
display(df_parallel[['problem_size', 'threads', 'fit_time', 'total_time']])

### 2.2. Speedup e Eficiência Paralela

In [ ]:
df_ref_o0 = df_baseline[df_baseline['opt_level']=='O0 (Raw)'][['samples','features','fit_time']]\
    .rename(columns={'fit_time':'t_ref_o0'})
df_ref_o3 = df_baseline[df_baseline['opt_level']=='O3 (Opt)'][['samples','features','fit_time']]\
    .rename(columns={'fit_time':'t_ref_o3'})

df_speedup = pd.merge(df_parallel, df_ref_o0, on=['samples','features'])
df_speedup = pd.merge(df_speedup, df_ref_o3, on=['samples','features'])

df_speedup['Speedup (vs O0)'] = df_speedup['t_ref_o0'] / df_speedup['fit_time']
df_speedup['Speedup (vs O3)'] = df_speedup['t_ref_o3'] / df_speedup['fit_time']

# Eficiência Paralela = Speedup / Threads
# Ideal = 1.0 (cada thread contribui proporcionalmente)
# Queda indica overhead de sincronização, contenção de memória, etc.
df_speedup['Parallel Efficiency'] = df_speedup['Speedup (vs O3)'] / df_speedup['threads']

print("✅ Speedup e Eficiência calculados.")
display(df_speedup[['problem_size','threads','Speedup (vs O3)','Parallel Efficiency']])

In [ ]:
df_filtered  = df_speedup[~df_speedup['problem_size'].str.contains('10000 x 15|100000 x 70')].copy()

df_plot = df_filtered.melt(
    id_vars=['problem_size','threads','samples','features'],
    value_vars=['Speedup (vs O0)','Speedup (vs O3)'],
    var_name='Referência', value_name='Speedup'
)

speedup_chart = alt.Chart(df_plot).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads'),
    y=alt.Y('Speedup:Q', scale=alt.Scale(type='log'), title='Speedup (Escala Log)'),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','threads','Speedup']
).properties(width=350, height=400, title='Escalabilidade Paralela').facet(
    column=alt.Column('Referência:N', title='Baseline de Referência')
).resolve_scale(y='independent')

speedup_chart.interactive()

In [ ]:
# Gráfico de Eficiência Paralela
# Linha ideal = 1.0 (escala perfeita)
# Cruzar abaixo de 0.5 indica que cada thread adicional
# está gerando mais overhead do que ganho.

ideal_eff = pd.DataFrame({'threads': [1, 40], 'Parallel Efficiency': [1.0, 1.0]})
ideal_line = alt.Chart(ideal_eff).mark_line(
    strokeDash=[5,5], color='gray', opacity=0.6
).encode(x='threads:Q', y='Parallel Efficiency:Q')

eff_chart = alt.Chart(df_filtered).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads'),
    y=alt.Y('Parallel Efficiency:Q', title='Eficiência (1.0 = ideal)', scale=alt.Scale(domain=[0, 1.1])),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','threads','Parallel Efficiency','Speedup (vs O3)']
).properties(width=600, height=400, title='Eficiência Paralela vs Número de Threads')

(eff_chart + ideal_line).interactive()

### 2.3. Hotspot (Paralelo)

In [ ]:
hot_para_metrics = ['CPU Time', 'Elapsed Time', 'Instructions Retired', 'CPI Rate']

hot_para_list = []
for _, row in df_parallel.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    t     = int(row['threads'])
    fpath = f"{DIR_PARA_VTUNE}/hot_{ds_id}_t{t}.csv"
    m_data = get_metrics_from_csv(fpath, hot_para_metrics)
    top_func, top_func_time = get_hotspot_function(fpath)
    hot_para_list.append({
        'problem_size': row['problem_size'],
        'threads':      t,
        'top_function': top_func,
        'func_cpu_time': top_func_time,
        **m_data
    })

df_hot_para = pd.DataFrame(hot_para_list)
df_hot_para['Core Utilization Factor'] = \
    df_hot_para['CPU Time'] / (df_hot_para['Elapsed Time'] * df_hot_para['threads'])

print("✅ Hotspots paralelos carregados.")
display(df_hot_para[['problem_size','threads','CPU Time','Elapsed Time','CPI Rate']].head(10))

In [ ]:
util_chart = alt.Chart(df_hot_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads'),
    y=alt.Y('Core Utilization Factor:Q', title='Fator de Utilização (1.0 = Ideal)'),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size','threads','Core Utilization Factor','CPU Time']
).properties(width=600, height=400, title='Eficiência do Escalonamento OpenMP')

util_chart.interactive()

### 2.4. HPC (Paralelo)

In [ ]:
# Coleta HPC Paralelo - Corrigida
hpc_para_targets = ['CPI Rate', 'Memory Bound', 'Cache Bound']

para_hpc_list = []
for _, row in df_parallel.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    t     = int(row['threads'])
    fpath = f"{DIR_PARA_VTUNE}/hpc_{ds_id}_t{t}.csv"
    
    m_data = get_hpc_metrics_isolated(fpath, hpc_para_targets)
    bw_data = get_dram_bandwidth_from_hpc(fpath) # Adicionado
    
    para_hpc_list.append({
        'problem_size': row['problem_size'],
        'threads':      t,
        'samples':      row['samples'],
        'features':     row['features'],
        **m_data,
        **bw_data
    })

df_hpc_para = pd.DataFrame(para_hpc_list)
print("✅ Métricas HPC paralelas carregadas (com GB/s reais).")
display(df_hpc_para[['problem_size','threads','CPI Rate','Memory Bound','Cache Bound','DRAM GB/s Average']].head(10))

In [ ]:
# Gráficos HPC Paralelos - Corrigidos
order = df_hpc_para.sort_values(['samples','features'])['problem_size'].unique().tolist()

mem_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Threads'),
    y=alt.Y('Memory Bound:Q', title='Memory Bound (%)'),
    color=alt.Color('problem_size:N', sort=order, title='Tamanho'),
    tooltip=['problem_size','threads','Memory Bound','DRAM GB/s Average']
).properties(width=380, height=300, title='Memory Bound por Thread')

# ATUALIZAÇÃO: Gráfico de Bandwidth em GB/s
dram_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Threads'),
    y=alt.Y('DRAM GB/s Average:Q', title='DRAM Average (GB/s)'),
    color=alt.Color('problem_size:N', sort=order, title='Tamanho'),
    tooltip=['problem_size','threads','DRAM GB/s Average']
).properties(width=380, height=300, title='Saturação de RAM (GB/s)')

cpi_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Threads'),
    y=alt.Y('CPI Rate:Q', title='CPI Rate'),
    color=alt.Color('problem_size:N', sort=order, title='Tamanho'),
    tooltip=['problem_size','threads','CPI Rate']
).properties(width=380, height=300, title='CPI Rate por Thread')

((mem_chart | dram_chart) & cpi_chart).resolve_scale(y='independent')